# Signal Analysis Walkthrough v2: Single Long Record With An Unlabeled Anomaly

This walkthrough uses one realistic CSV file with two columns: `time` and `voltage`. The scenario is a high-frequency vibration/acoustic sensor trace from one operating run. There are no labels and no per-event annotations; the task is to find suspicious time regions from signal behavior alone.

The CSV is synthetic but shaped like a practical sensor export: uniformly sampled time values, voltage amplitudes, rotating-machine tones, broadband noise, slow operating drift, and one short abnormal interval. The generator is saved at `scripts/generate_vibration_anomaly_csv.py` so the data recipe is traceable and reproducible.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from signal_processing_prep.data_loading import SignalDatasetLoader
from signal_processing_prep.features import FeatureExtractor, FrequencyBand, SlidingWindowConfig
from signal_processing_prep.modeling import RobustZScoreScorer
from signal_processing_prep.plotting import plot_frequency_spectrum, plot_spectrogram_dynamic_range, plot_time_signal, plot_time_signal_adaptive, plot_wavelet_scalogram
from signal_processing_prep.quality import assess_signal_quality
from signal_processing_prep.records import SignalRecord

In [ ]:
from IPython import get_ipython

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic("matplotlib", "widget")
        print("Configured matplotlib for interactive widget backend.")
    except Exception:
        ip.run_line_magic("matplotlib", "inline")
        print("Interactive widget backend unavailable; using inline backend.")

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "vibration_anomaly_single_record.csv"
DATA_PATH

## 2. Load The CSV Through The Project Loader

The CSV does not provide `sampling_rate_hz` directly. The loader infers it from the `time` column and stores timing observations in `record.acquisition`.

In [ ]:
record = SignalDatasetLoader().load_file(DATA_PATH, signal_column="voltage")

print(record)
print(f"Samples: {record.n_samples:,}")
print(f"Duration: {record.duration_seconds:.2f} s")
print(f"Sampling rate: {record.sampling_rate_hz:.1f} Hz")
print("Acquisition diagnostics:")
for key in [
    "sampling_rate_source",
    "time_column",
    "time_step_median_seconds",
    "time_step_jitter_fraction",
    "time_gap_count",
]:
    print(f"  {key}: {getattr(record.acquisition, key)}")

## 3. Quality Check Before DSP

Before searching for patterns, confirm that the record is finite, long enough, and close enough to uniformly sampled for FFT-based analysis.

In [ ]:
quality = assess_signal_quality(record)
pd.DataFrame([quality.to_dict()])

## 4. Inspect The Raw Time Signal

A long high-frequency trace is hard to read sample-by-sample, so the overview plot uses adaptive display downsampling. The visible line is recomputed from the original record when the interactive x-axis range changes.

In [ ]:
fig, ax = plot_time_signal_adaptive(record, max_points=2000)
ax.set_ylabel("Voltage [V]");

## 5. Global Frequency View

The full-record PSD shows the ordinary operating components and any broadband or resonant content. This is useful context, but a short anomaly can be diluted in a global spectrum.

In [ ]:
fig, ax = plot_frequency_spectrum(
    record,
    spectrum_type="psd",
   # max_frequency_hz=6000,
    nperseg=4*4096,
)
ax.set_title("Full-record PSD overview")
#ax.set_xscale("log")
ax.set_yscale("log")  

## 6. Time-Frequency View

A spectrogram is a better first tool when the abnormal behavior may be localized in time. Here `scipy.signal.spectrogram(..., mode="psd")` is STFT-like: the signal is windowed in time and Fourier-transformed per window, but the plotted values are power spectral density rather than complex STFT coefficients or amplitude. The frequency axis uses a true log scale for inspection, so the 0 Hz/DC bin is omitted from the display.

A Morlet wavelet scalogram gives a complementary time-frequency view. The spectrogram uses one fixed analysis window, while the scalogram is scale-based: low frequencies use longer wavelets and higher-frequency transients are more localized in time.

In [ ]:
fig, ax = plot_spectrogram_dynamic_range(
    record,
    window_seconds=0.1,
    step_seconds=0.05,
    max_frequency_hz=6000,
    frequency_scale="log",
)
ax.set_title("Full-record spectrogram")

## 7. Sliding-Window Features

Turn the single long record into overlapping windows. Each window becomes one observation for unsupervised anomaly scoring. The bands are generic engineering guesses: low-frequency rotating components, a mid/high resonance range, and broadband high-frequency energy.

In [ ]:
window_config = SlidingWindowConfig(
    window_seconds=0.20,
    step_seconds=0.025,
    frequency_bands=(
        FrequencyBand("rotating_40_500", 40.0, 500.0),
        FrequencyBand("resonance_1800_3200", 1800.0, 3200.0),
        FrequencyBand("broadband_3200_6000", 3200.0, 6000.0),
    ),
    frequency_window="hann",
    normalize_frequency_window_power=True,
)

feature_table = FeatureExtractor().extract_windows(record, window_config)
features = feature_table.to_dataframe()
features.head()

## 8. Rank Suspicious Windows

Use robust z-scores so the ranking is not dominated by ordinary operating variation. This is deliberately simple and explainable: windows are suspicious when their RMS, crest factor, and high-frequency band energies are unusually high relative to the rest of the run.

In [ ]:
score_columns = [
    "rms",
    "crest_factor",
    "band_energy_resonance_1800_3200",
    "band_energy_broadband_3200_6000",
]

ranked = RobustZScoreScorer(feature_columns=tuple(score_columns)).score(feature_table).prediction_frame.sort_values("anomaly_score", ascending=False)
ranked[[
    "window_start_seconds",
    "window_end_seconds",
    "window_center_seconds",
    "anomaly_score",
]].head(50)

## 9. Plot The Anomaly Score Over Time

The highest-scoring windows define a candidate region to inspect, not a final diagnosis.

In [ ]:
top_window = ranked.iloc[0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ranked["window_center_seconds"], ranked["anomaly_score"], linewidth=1.2)
ax.axvspan(
    top_window["window_start_seconds"],
    top_window["window_end_seconds"],
    color="tab:red",
    alpha=0.2,
    label="top candidate window",
)
ax.set_title("Sliding-window anomaly score")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Robust anomaly score")
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()

print(
    "Top candidate window: "
    f"{top_window['window_start_seconds']:.3f} s to "
    f"{top_window['window_end_seconds']:.3f} s"
)

## 10. Zoom Into The Candidate Region

Now inspect the raw waveform and localized spectrogram around the highest-scoring region.

In [ ]:
candidate_center = float(top_window["window_center_seconds"])
zoom_start = max(candidate_center - 0.6, 0.0)
zoom_duration = 1.2

fig, ax = plot_time_signal(
    record,
    start_seconds=zoom_start,
    duration_seconds=zoom_duration,
    max_points=8000,
)
ax.axvspan(
    float(top_window["window_start_seconds"]),
    float(top_window["window_end_seconds"]),
    color="tab:red",
    alpha=0.2,
)
ax.set_ylabel("Voltage [V]")
ax.set_title("Raw signal around top candidate");

In [ ]:
start_index = int(round(zoom_start * record.sampling_rate_hz))
end_index = int(round((zoom_start + zoom_duration) * record.sampling_rate_hz))
end_index = min(end_index, record.n_samples)

zoom_record = record.segment(start_index, end_index, index=0)

fig, ax = plot_spectrogram_dynamic_range(
    zoom_record,
    window_seconds=0.05,
    step_seconds=0.0025,
    max_frequency_hz=6000,
    frequency_scale="log",
)
ax.set_title("Candidate-region spectrogram; time is relative to zoom window");

Use the same wavelet view on the candidate region for a more localized comparison against the fixed-window spectrogram.

In [ ]:
fig, ax = plot_wavelet_scalogram(
    zoom_record,
    min_frequency_hz=20.0,
    max_frequency_hz=6000.0,
    n_frequencies=128,
    frequency_scale="log",
)
ax.set_title("Candidate-region Morlet wavelet scalogram; time is relative to zoom window");

## 11. Interpretation

This is the intended interview story for the v2 walkthrough:

1. The CSV contains one unlabeled high-frequency sensor record.
2. The loader infers sampling rate from timestamps and records timing assumptions.
3. Quality checks confirm whether FFT-based methods are valid enough to use.
4. Global plots establish ordinary operating content but do not localize the event.
5. Sliding-window features convert one long record into comparable time regions.
6. A simple robust anomaly score highlights a short candidate interval.
7. The final claim is not a class label; it is a prioritized region for engineering inspection.